# 73. Set Matrix Zeroes
**Difficulty:** 🟡 Medium · **Topic:** Matrix · **LeetCode:** https://leetcode.com/problems/set-matrix-zeroes/

## 💡 Concepts

**Core concept(s):** Use the matrix's own **first row and column as markers** to record which rows/cols must be zeroed — O(1) extra space.

**Why it applies here:** If a cell is 0, its whole row and column become 0. Remembering that in separate sets costs O(m+n); instead, store the flags in the first row/column of the matrix itself, handling those two lines specially.

**Key intuition:** Mark the rows/cols to zero in the border cells, then apply them in a second pass.

---

### 📚 Working with a Matrix (Grid)
A matrix is a list of rows. Common moves: walk with `(r, c)` coordinates, **transpose** (swap rows/columns), or use the first row/column as scratch space to save memory.

---

**Prerequisite knowledge:**
- In-place marking.
- Careful two-pass order.

## 📝 Problem

If an element is 0, set its entire row and column to 0. Do it **in place**.

**Example**
```
[[1,1,1],[1,0,1],[1,1,1]] -> [[1,0,1],[0,0,0],[1,0,1]]
```

> Two approaches: extra sets `O(m+n)` space and first-row/col markers `O(1)` space.

### Approach 1 — Remember Rows/Cols (worst on memory)

**Idea:** Record which rows and columns contain a 0, then zero them.

**Time:** `O(m×n)`. **Space:** `O(m+n)`.

In [ ]:
def set_zeroes_sets(matrix):
    rows, cols = set(), set()              # remember which rows / columns contain a 0
    m, n = len(matrix), len(matrix[0])
    for r in range(m):
        for c in range(n):
            if matrix[r][c] == 0:
                rows.add(r); cols.add(c)   # this row and column must be zeroed
    for r in range(m):
        for c in range(n):
            if r in rows or c in cols:     # zero any cell in a marked row or column
                matrix[r][c] = 0
    return matrix

### Approach 2 — First Row/Col as Markers (optimal)

**Idea:** Use row 0 and column 0 to flag zeros. Track the first row/col separately, mark, apply, then handle the borders last.

**Time:** `O(m×n)`. **Space:** `O(1)`.

In [ ]:
def set_zeroes_inplace(matrix):
    m, n = len(matrix), len(matrix[0])
    first_row = any(matrix[0][c] == 0 for c in range(n))   # does row 0 itself need zeroing?
    first_col = any(matrix[r][0] == 0 for r in range(m))   # does column 0 itself need zeroing?
    for r in range(1, m):                  # use row 0 / column 0 as notepads for the rest
        for c in range(1, n):
            if matrix[r][c] == 0:
                matrix[r][0] = 0; matrix[0][c] = 0   # mark this row and column in the borders
    for r in range(1, m):
        for c in range(1, n):
            if matrix[r][0] == 0 or matrix[0][c] == 0:   # marked -> zero the cell
                matrix[r][c] = 0
    if first_row:                          # finally handle the border row/column themselves
        for c in range(n): matrix[0][c] = 0
    if first_col:
        for r in range(m): matrix[r][0] = 0
    return matrix

In [ ]:
# Correctness check
def clone(g): return [row[:] for row in g]
tests = [
    ([[1,1,1],[1,0,1],[1,1,1]], [[1,0,1],[0,0,0],[1,0,1]]),
    ([[0,1,2,0],[3,4,5,2],[1,3,1,5]], [[0,0,0,0],[0,4,5,0],[0,3,1,0]]),
]
for g, exp in tests:
    assert set_zeroes_sets(clone(g)) == exp
    assert set_zeroes_inplace(clone(g)) == exp
    print("OK", exp)
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    grid = [[1] * n for _ in range(n)]
    grid[n // 2][n // 2] = 0
    return (grid,)
solutions = {
    "sets    O(m+n) space": set_zeroes_sets,
    "in-place O(1) space ": set_zeroes_inplace,
}
sizes = [100, 200, 400, 800]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Use the input as scratch space:** border cells store the flags → O(1) extra memory.
- **Handle the special lines last:** the marker row/col must be applied after everything else.
- **Signal:** "modify a grid in place with O(1) space".
- **Related problems:** Game of Life, Rotate Image, Spiral Matrix.
- **Common pitfalls:** (1) overwriting markers before using them; (2) forgetting the first row/col edge cases.